In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from dateutil import parser as date_parser
import time

# Configuration
START_DATE = datetime(2026, 1, 21)
END_DATE = datetime(2026, 2, 1)
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'}

def scrape_wpln(keyword):
    """Scrapes WPLN (Nashville Public Radio) search results."""
    results = []
    search_url = f"https://wpln.org/?s={keyword}"
    try:
        response = requests.get(search_url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        for item in soup.find_all('article'):
            link_tag = item.find('a', href=True)
            date_tag = item.find('time') or item.find(class_='date')
            
            if link_tag and date_tag:
                url = link_tag['href']
                headline = link_tag.get_text().strip()
                pub_date = date_parser.parse(date_tag.get_text().strip())

                if START_DATE <= pub_date <= END_DATE:
                    print(f"Scraping WPLN: {headline}")
                    content = extract_article_body(url, "div.entry-content")
                    results.append(create_row(url, headline, content, pub_date, "WPLN"))
    except Exception as e:
        print(f"Error scraping WPLN: {e}")
    return results

def extract_article_body(url, selector):
    """Generic helper to pull main text from an article."""
    try:
        time.sleep(1) # Be kind to servers
        res = requests.get(url, headers=HEADERS, timeout=10)
        s = BeautifulSoup(res.text, 'html.parser')
        main_div = s.select_one(selector)
        return main_div.get_text(separator=' ', strip=True) if main_div else "Content not found."
    except:
        return "Failed to retrieve."

def create_row(link, headline, content, date_pub, publisher):
    """Formats the data into a dictionary for the DataFrame."""
    return {
        "link": link,
        "headline": headline,
        "full contents": content,
        "date published": date_pub.strftime('%Y-%m-%d') if date_pub else "N/A",
        "date scraped": datetime.now().strftime('%Y-%m-%d'),
        "publisher": publisher
    }

# Execute
all_data = []
all_data.extend(scrape_wpln("storm"))

df_wpln = pd.DataFrame(all_data)

# Initialize shared final DataFrame once
if "final_results_df" not in globals():
    final_results_df = pd.DataFrame(columns=[
        "link",
        "headline",
        "full contents",
        "date published",
        "date scraped",
        "publisher",
    ])

# Append WPLN rows into shared final DataFrame
if not df_wpln.empty:
    final_results_df = pd.concat([final_results_df, df_wpln], ignore_index=True)

# Show the results
print(f"\nCollected {len(df_wpln)} WPLN articles.")
print(df_wpln[['publisher', 'headline', 'date published']].head())
print(f"Running total in final_results_df: {len(final_results_df)}")

Scraping WPLN: Winter storm week 1: Tennessee fatalities rise; outage frustrations mounting
Scraping WPLN: Winter storm week 2: Challenges linger and some school districts remain closed
Scraping WPLN: ‘Once in a generation’ ice storm has Nashvillians remembering 1994 disaster
Scraping WPLN: Comfort from the cold: Acts of kindness help Nashvillians through the storm
Scraping WPLN: Middle Tennessee is bracing for a major winter storm this weekend. Here’s what to expect.

Collected 5 WPLN articles.
  publisher                                           headline date published
0      WPLN  Winter storm week 1: Tennessee fatalities rise...     2026-01-24
1      WPLN  Winter storm week 2: Challenges linger and som...     2026-02-01
2      WPLN  ‘Once in a generation’ ice storm has Nashvilli...     2026-01-28
3      WPLN  Comfort from the cold: Acts of kindness help N...     2026-01-28
4      WPLN  Middle Tennessee is bracing for a major winter...     2026-01-21
Running total in final_results_

In [3]:
# Optional checkpoint (shared DataFrame)
if "final_results_df" in globals() and not final_results_df.empty:
    print(f"Checkpoint rows in final_results_df: {len(final_results_df)}")
else:
    print("final_results_df is empty right now.")

Checkpoint rows in final_results_df: 5


In [4]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Robust Session Setup
def get_secure_session():
    session = requests.Session()
    # Retry strategy: 3 retries, with increasing wait times (1s, 2s, 4s)
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    
    # Comprehensive headers to look like a standard Nashville-based user
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
        'Referer': 'https://www.google.com/',
        'DNT': '1'
    })
    return session

def scrape_tema():
    session = get_secure_session()
    hub_url = "https://www.tn.gov/tema/updates/2026-disasters/january-2026-winter-weather.html"
    results = []
    
    try:
        # Increased timeout to 20 seconds for slow gov servers
        response = session.get(hub_url, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # TEMA's 2026 layout uses a list for Flash Reports
        for link in soup.select('a[href*="flash-report"], a[href*="news-release"]'):
            title = link.get_text(strip=True)
            url = link['href']
            if not url.startswith('http'):
                url = "https://www.tn.gov" + url
            
            # Extract content from each individual report
            print(f"Fetching TEMA detail: {title[:30]}...")
            detail_res = session.get(url, timeout=20)
            detail_soup = BeautifulSoup(detail_res.text, 'html.parser')
            
            # The body is usually in .article-content or .main-content
            body = detail_soup.select_one('.article-content, .main-content, article')
            content = body.get_text(separator=' ', strip=True) if body else "Content not found."
            
            results.append({
                "link": url,
                "headline": title,
                "full contents": content,
                "date published": "January 2026", # You can parse specific dates from the title
                "date scraped": datetime.now().strftime('%Y-%m-%d'),
                "publisher": "TEMA"
            })
    except Exception as e:
        print(f"Connection failed: {e}")
    
    return pd.DataFrame(results)

# Run it
df_tema = scrape_tema()

# Initialize shared final DataFrame if needed
if "final_results_df" not in globals():
    final_results_df = pd.DataFrame(columns=[
        "link",
        "headline",
        "full contents",
        "date published",
        "date scraped",
        "publisher",
    ])

# Append TEMA rows into shared final DataFrame
if not df_tema.empty:
    final_results_df = pd.concat([final_results_df, df_tema], ignore_index=True)

print(df_tema.head())
print(f"Running total in final_results_df: {len(final_results_df)}")

Fetching TEMA detail: News Releases...
Fetching TEMA detail: Flash Report #16 - Winter Weat...
Fetching TEMA detail: Flash Report #15 - Winter Weat...
Fetching TEMA detail: Flash Report #14 - Winter Weat...
Fetching TEMA detail: Flash Report #13 - Winter Weat...
Fetching TEMA detail: Flash Report #12 - Winter Weat...
Fetching TEMA detail: Flash Report #11 - Winter Weat...
Fetching TEMA detail: Flash Report #10 - Winter Weat...
Fetching TEMA detail: Flash Report #9 Winter Weather...
Fetching TEMA detail: Flash Report #8 - Winter Weath...
Fetching TEMA detail: Flash Report #7 - Winter Weath...
Fetching TEMA detail: Flash Report #6 - Winter Weath...
Fetching TEMA detail: Flash Report #5 - Winter Weath...
Fetching TEMA detail: Flash Report #4 - Winter Weath...
Fetching TEMA detail: Flash Report #3 - Winter Weath...
Fetching TEMA detail: Flash Report #2 - Winter Weath...
Fetching TEMA detail: Flash Report #1 - Winter Weath...
                                                link  \
0  https:

In [5]:
# Optional checkpoint (shared DataFrame)
if "final_results_df" in globals() and not final_results_df.empty:
    print(f"Checkpoint rows in final_results_df: {len(final_results_df)}")
else:
    print("final_results_df is empty right now.")

Checkpoint rows in final_results_df: 22


In [7]:
# import requests
# import pandas as pd

# def get_wayback_links(domain, start_date, end_date):
#     # CDX API Query
#     api_url = f"http://web.archive.org/cdx/search/cdx?url={domain}/*&from={start_date}&to={end_date}&output=json&fl=original,timestamp"
    
#     response = requests.get(api_url)
#     data = response.json()
    
#     # Skip the header row and create DataFrame
#     df = pd.DataFrame(data[1:], columns=data[0])
#     # Filter for articles (usually containing /post/ or /article/ or a date string)
#     df = df[df['original'].str.contains('2026', na=False)]
#     return df

# # Example: Get all WPLN URLs from the storm week
# links = ["wpln.org","tennesseelookout.com","nashvillescene.com/news","nashville.gov/news"]
# df_links = get_wayback_links(, "20260121", "20260201")

# print(df_links.head())

In [9]:
import time
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dateutil import parser as date_parser

# --- Configuration ---
START_DATE = datetime(2026, 1, 21)
END_DATE = datetime(2026, 2, 1)
KEYWORDS = "storm"
KEYWORDS_LOWER = KEYWORDS.lower()
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/122.0.0.0"}


def create_row(link, headline, content, date_pub, publisher):
    return {
        "link": link,
        "headline": headline,
        "full contents": content,
        "date published": date_pub.strftime("%Y-%m-%d") if isinstance(date_pub, datetime) else str(date_pub),
        "date scraped": datetime.now().strftime("%Y-%m-%d"),
        "publisher": publisher,
    }


def clean_datetime(dt):
    if isinstance(dt, datetime) and dt.tzinfo is not None:
        return dt.replace(tzinfo=None)
    return dt


def parse_date_safe(raw):
    if not raw:
        return None
    try:
        return clean_datetime(date_parser.parse(str(raw).strip()))
    except Exception:
        return None


def in_date_range(pub_date):
    return isinstance(pub_date, datetime) and START_DATE <= pub_date <= END_DATE


def get_content(url, selectors):
    try:
        time.sleep(1)  # Politeness
        res = requests.get(url, headers=HEADERS, timeout=15)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")

        for selector in selectors:
            body = soup.select_one(selector)
            if body:
                for bad in body.select("script, style, aside, noscript"):
                    bad.decompose()
                text = body.get_text(separator=" ", strip=True)
                if len(text) > 80:
                    return text
        return "No content found."
    except Exception:
        return "Extraction failed."


def extract_html_to_text(raw_html):
    if not raw_html:
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    for bad in soup.select("script, style, aside, noscript"):
        bad.decompose()
    return soup.get_text(separator=" ", strip=True)


# --- Site Specific Logic ---

def scrape_nes():
    """Nashville Electric Service Newsroom via RSS search feed."""
    results = []
    url = f"https://nespowernews.com/?s={KEYWORDS}&feed=rss2"
    feed = BeautifulSoup(requests.get(url, headers=HEADERS, timeout=20).text, "xml")

    for item in feed.find_all("item"):
        link_tag = item.find("link")
        title_tag = item.find("title")
        date_tag = item.find("pubDate")
        if not (link_tag and title_tag and date_tag):
            continue

        link = link_tag.get_text(strip=True)
        headline = title_tag.get_text(" ", strip=True)
        pub_date = parse_date_safe(date_tag.get_text(" ", strip=True))
        if not in_date_range(pub_date):
            continue

        content = get_content(link, [".post-content", ".entry-content", "article"])
        results.append(create_row(link, headline, content, pub_date, "NES Power News"))
    return results


def scrape_tn_lookout():
    """Tennessee Lookout via RSS (search endpoint is bot-protected)."""
    results = []
    url = "https://tennesseelookout.com/feed/"
    feed = BeautifulSoup(requests.get(url, headers=HEADERS, timeout=20).text, "xml")

    for item in feed.find_all("item"):
        title_tag = item.find("title")
        link_tag = item.find("link")
        date_tag = item.find("pubDate")
        desc_tag = item.find("description")
        content_tag = item.find("content:encoded")
        if not (title_tag and link_tag and date_tag):
            continue

        headline = title_tag.get_text(" ", strip=True)
        description = desc_tag.get_text(" ", strip=True) if desc_tag else ""
        raw_content = content_tag.get_text(" ", strip=True) if content_tag else ""
        if KEYWORDS_LOWER not in f"{headline} {description} {raw_content}".lower():
            continue

        pub_date = parse_date_safe(date_tag.get_text(" ", strip=True))
        if not in_date_range(pub_date):
            continue

        content = extract_html_to_text(raw_content) or extract_html_to_text(description) or "No content found."
        results.append(create_row(link_tag.get_text(strip=True), headline, content, pub_date, "TN Lookout"))
    return results


def scrape_nashville_scene():
    """Nashville Scene via RSS search, then article page body extraction."""
    results = []
    url = f"https://www.nashvillescene.com/search/?q={KEYWORDS}&t=article&f=rss"
    feed = BeautifulSoup(requests.get(url, headers=HEADERS, timeout=20).text, "xml")

    for item in feed.find_all("item"):
        link_tag = item.find("link")
        title_tag = item.find("title")
        date_tag = item.find("pubDate")
        if not (link_tag and title_tag and date_tag):
            continue

        link = link_tag.get_text(strip=True)
        headline = title_tag.get_text(" ", strip=True)
        pub_date = parse_date_safe(date_tag.get_text(" ", strip=True))
        if not in_date_range(pub_date):
            continue

        content = get_content(link, [".asset-body", ".asset-content", "article.asset"])
        results.append(create_row(link, headline, content, pub_date, "Nashville Scene"))
    return results


def scrape_metro_gov():
    """Metro Nashville Mayor newsroom (paginated listing)."""
    results = []
    page = 0
    base_url = "https://www.nashville.gov/departments/mayor/news"

    while page < 8:
        page_url = base_url if page == 0 else f"{base_url}?page={page}"
        soup = BeautifulSoup(requests.get(page_url, headers=HEADERS, timeout=20).text, "html.parser")
        rows = soup.select(".views-row")
        if not rows:
            break

        hit_older_than_window = False
        for art in rows:
            link_tag = art.select_one("a[href]")
            date_tag = art.select_one("time")
            if not (link_tag and date_tag):
                continue

            pub_date = parse_date_safe(date_tag.get_text(" ", strip=True))
            if not isinstance(pub_date, datetime):
                continue
            if pub_date < START_DATE:
                hit_older_than_window = True
                continue
            if pub_date > END_DATE:
                continue

            link = urljoin("https://www.nashville.gov", link_tag["href"])
            headline = link_tag.get_text(" ", strip=True)
            content = get_content(link, [".field--name-body", "article"])
            if KEYWORDS_LOWER not in f"{headline} {content}".lower():
                continue

            results.append(create_row(link, headline, content, pub_date, "Metro Nashville"))

        if hit_older_than_window:
            break
        page += 1

    return results


# --- Main Pipeline ---

all_results = []
all_results.extend(scrape_nashville_scene())  # iterate Scene first
all_results.extend(scrape_nes())
all_results.extend(scrape_tn_lookout())
all_results.extend(scrape_metro_gov())

# De-duplicate by URL while preserving order
seen = set()
deduped = []
for row in all_results:
    if row["link"] in seen:
        continue
    seen.add(row["link"])
    deduped.append(row)

df_multi_source = pd.DataFrame(deduped)

# Initialize shared final DataFrame if needed
if "final_results_df" not in globals():
    final_results_df = pd.DataFrame(columns=[
        "link",
        "headline",
        "full contents",
        "date published",
        "date scraped",
        "publisher",
    ])

# Append this multi-source batch into shared final DataFrame
if not df_multi_source.empty:
    final_results_df = pd.concat([final_results_df, df_multi_source], ignore_index=True)

# Verification
print(f"Scraped {len(df_multi_source)} articles related to '{KEYWORDS}'.")
if not df_multi_source.empty:
    print(df_multi_source[["publisher", "headline", "date published"]])
    print("\nContent length by article:")
    print(df_multi_source[["publisher", "headline", "full contents"]].assign(content_len=df_multi_source["full contents"].str.len())[["publisher", "headline", "content_len"]])

print(f"Running total in final_results_df: {len(final_results_df)}")

Scraped 7 articles related to 'storm'.
         publisher                                           headline  \
0   NES Power News                           Latest Storm Restoration   
1   NES Power News  Winter Storm Expected in Music City: Important...   
2  Metro Nashville  Mayor Freddie O'Connell, Office of Emergency M...   
3  Metro Nashville  Mayor Freddie O'Connell, Office of Emergency M...   
4  Metro Nashville  Mayor Freddie O'Connell and Office of Emergenc...   
5  Metro Nashville  Mayor Freddie O'Connell and Office of Emergenc...   
6  Metro Nashville  Mayor Freddie O'Connell, Office of Emergency M...   

  date published  
0     2026-01-29  
1     2026-01-22  
2     2026-01-31  
3     2026-01-29  
4     2026-01-28  
5     2026-01-26  
6     2026-01-26  

Content length by article:
         publisher                                           headline  \
0   NES Power News                           Latest Storm Restoration   
1   NES Power News  Winter Storm Expected in Music

In [10]:
# Final unified save
if "final_results_df" not in globals() or final_results_df.empty:
    print("No data available in final_results_df. Run scraper cells first.")
else:
    # Normalize and de-duplicate final output
    final_results_df = final_results_df.copy()
    final_results_df["date published"] = final_results_df["date published"].astype(str)
    final_results_df = final_results_df.drop_duplicates(subset=["link"], keep="first")

    final_results_df.to_csv("final_results.csv", index=False)
    print(f"Saved {len(final_results_df)} rows to final_results.csv")
    print(final_results_df[["publisher", "headline", "date published"]].head())

Saved 29 rows to final_results.csv
  publisher                                           headline date published
0      WPLN  Winter storm week 1: Tennessee fatalities rise...     2026-01-24
1      WPLN  Winter storm week 2: Challenges linger and som...     2026-02-01
2      WPLN  ‘Once in a generation’ ice storm has Nashvilli...     2026-01-28
3      WPLN  Comfort from the cold: Acts of kindness help N...     2026-01-28
4      WPLN  Middle Tennessee is bracing for a major winter...     2026-01-21
